In [ ]:
!pip install underthesea

In [ ]:
import json
import time
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from underthesea import word_tokenize

# --- HÀM HỖ TRỢ LOAD DATA ---
def load_financial_dataset(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    df = pd.DataFrame(data)
    return df['sentence'].astype(str).tolist(), df['sentiment'].tolist()

# --- HÀM TIỀN XỬ LÝ TIẾNG VIỆT ---
def vi_preprocessor(text):
    return word_tokenize(text, format="text")

# --- HÀM ERROR ANALYSIS ---
def analyze_errors(X_text, y_true, y_pred, num_samples=3):
    print("\n--- BẮT ĐẦU BƯỚC ERROR ANALYSIS ---")
    X_text = np.array(X_text)
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    error_indices = np.where(y_true != y_pred)[0]
    total_errors = len(error_indices)
    total_samples = len(y_true)
    error_rate = (total_errors / total_samples) * 100

    print(f"Tổng số mẫu bị đoán sai: {total_errors}/{total_samples} ({error_rate:.2f}%)")

    labels = sorted(list(set(y_true)))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    cm_df = pd.DataFrame(cm, index=[f"Actual_{l}" for l in labels], columns=[f"Pred_{l}" for l in labels])
    print("\nMa trận nhầm lẫn (Confusion Matrix):")
    print(cm_df)

    print(f"\nChi tiết một số mẫu lỗi điển hình (Tối đa {num_samples} mẫu cho mỗi cặp lỗi):")
    for actual in labels:
        for pred in labels:
            if actual == pred:
                continue
            pair_indices = np.where((y_true == actual) & (y_pred == pred))[0]
            if len(pair_indices) > 0:
                print(f"\n[LỖI] Thực tế: '{actual}' --> Mô hình đoán: '{pred}' (Tổng cộng: {len(pair_indices)} mẫu)")
                print("-" * 80)
                sampled_indices = pair_indices[:num_samples]
                for idx in sampled_indices:
                    print(f"- Văn bản: {X_text[idx]}")
                print("-" * 80)

# --- HÀM HUẤN LUYỆN VÀ ĐÁNH GIÁ THỬ NGHIỆM ---
def run_baseline(train_path, test_path, language='en'):
    print(f"\n=== BẮT ĐẦU TRIỂN KHAI BASELINE: LOGISTIC REGRESSION + TF-IDF ({language.upper()}) ===")

    # 1. Load dữ liệu
    X_train, y_train = load_financial_dataset(train_path)
    X_test, y_test = load_financial_dataset(test_path)
    num_test_samples = len(X_test)
    print(f"Kích thước tập Train: {len(X_train)} | Kích thước tập Test: {num_test_samples}")

    # 2. Khởi tạo Vectorizer
    if language == 'vi':
        vectorizer = TfidfVectorizer(
            preprocessor=vi_preprocessor,
            ngram_range=(1, 2),
            max_features=10000,
            sublinear_tf=True
        )
    else:
        vectorizer = TfidfVectorizer(
            ngram_range=(1, 2),
            max_features=10000,
            sublinear_tf=True,
            stop_words='english'
        )

    # 3. Biến đổi dữ liệu Train và Huấn luyện mô hình
    X_train_vec = vectorizer.fit_transform(X_train)
    model = LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced', random_state=42)
    model.fit(X_train_vec, y_train)

    # 4. CHẠY TRÊN TẬP TEST VÀ ĐO THỜI GIAN
    # Bắt đầu đo bao gồm cả bước Vectorize tập test và bước Predict của mô hình
    start_time = time.perf_counter()

    X_test_vec = vectorizer.transform(X_test)
    y_pred = model.predict(X_test_vec)

    end_time = time.perf_counter()

    # Tính toán thời gian
    total_test_time = end_time - start_time  # Giây
    avg_time_per_sample_ms = (total_test_time / num_test_samples) * 1000  # Mili-giây

    print(f"\n--- BỘ ĐO THỜI GIAN CHẠY TẬP TEST ({language.upper()}) ---")
    print(f"Tổng thời gian xử lý tập Test: {total_test_time:.4f} giây")
    print(f"Thời gian xử lý trung bình: {avg_time_per_sample_ms:.4f} ms/mẫu")
    print("-" * 40)

    # 5. Dự đoán và đánh giá tổng quan
    print(f"\nKết quả đánh giá trên tập {language}_test.json:")
    print(classification_report(y_test, y_pred, digits=4))

    # 6. Error Analysis
    analyze_errors(X_test, y_test, y_pred, num_samples=3)

    return model, vectorizer

# --- THỰC THI PIPELINE ---
if __name__ == "__main__":
    en_train_file = "en_train.json"
    en_test_file = "en_test.json"

    try:
        en_model, en_vectorizer = run_baseline(en_train_file, en_test_file, language='en')
    except FileNotFoundError as e:
        print(f"Lỗi: Không tìm thấy file dữ liệu tiếng Anh. {e}")

    vi_train_file = "vi_train.json"
    vi_test_file = "vi_test.json"

    try:
        vi_model, vi_vectorizer = run_baseline(vi_train_file, vi_test_file, language='vi')
    except FileNotFoundError as e:
        print(f"Lỗi: Không tìm thấy file dữ liệu tiếng Việt. {e}")